In [2]:
import pandas as pd
import joblib
import numpy as np


In [3]:
df = pd.read_csv("../data/processed/telco_featured.csv")

pipeline = joblib.load("../models/churn_pipeline.pkl")
threshold = joblib.load("../models/threshold.pkl")

In [4]:
X = df.drop("Churn", axis=1)
df["churn_probability"] = pipeline.predict_proba(X)[:, 1]

In [5]:
df["target"] = df["churn_probability"] >= threshold
target_df = df[df["target"] == True]

In [6]:
RETENTION_COST = 500
RETENTION_SUCCESS_RATE = 0.3 

In [7]:
df["customer_value"] = df["MonthlyCharges"] * df["tenure"]

In [8]:
thresholds = np.linspace(0.1, 0.9, 20)

results = []

for t in thresholds:
    df["target"] = df["churn_probability"] >= t
    target_df = df[df["target"]]
    
    customers_targeted = len(target_df)
    true_positives = target_df[target_df["Churn"] == 1].shape[0]
    
    cost = customers_targeted * RETENTION_COST
    revenue = true_positives * df["customer_value"].mean() * RETENTION_SUCCESS_RATE
    
    roi = (revenue - cost) / cost if cost > 0 else 0
    
    results.append((t, customers_targeted, true_positives, roi))

roi_df = pd.DataFrame(results, columns=["threshold", "targeted", "true_positives", "roi"])
roi_df.sort_values(by="roi", ascending=False)

,threshold,targeted,true_positives,roi
19,0.900000,331,286,0.185446
18,0.857895,580,458,0.083381
17,0.815789,829,612,0.012839
16,0.773684,1166,823,-0.031622
15,0.731579,1468,994,-0.071025
14,0.689474,1752,1111,-0.129991
13,0.647368,2012,1219,-0.168773
12,0.605263,2266,1314,-0.204429
11,0.563158,2486,1394,-0.230683
10,0.521053,2730,1462,-0.265269


In [9]:

threshold = 0.7
df["target"] = (df["churn_probability"] >= threshold).astype(int)

In [10]:
# Expected Revenue Saved
df["expected_saved"] = (
    df["customer_value"] * RETENTION_SUCCESS_RATE * df["target"]
)

# Cost per customer (for ROI dashboard)
df["cost"] = df["target"] * RETENTION_COST

In [11]:
retention_rates = [0.2, 0.3, 0.4, 0.5]
costs = [200, 300, 500]

results = []

for rate in retention_rates:
    for cost_per_customer in costs:
        cost = customers_targeted * cost_per_customer
        revenue = true_positives * df["customer_value"].mean() * rate
        
        roi = (revenue - cost) / cost
        
        results.append((rate, cost_per_customer, roi))

sensitivity_df = pd.DataFrame(results, columns=["retention_rate", "cost", "roi"])
sensitivity_df.sort_values(by="roi", ascending=False)

print(f"Customers Targeted: {customers_targeted}")
print(f"True Positives: {true_positives}")
print(f"Total Cost: {cost}")
print(f"Revenue Saved: {revenue:.2f}")
print(f"ROI: {roi:.2f}")

Customers Targeted: 331
True Positives: 286
Total Cost: 165500
Revenue Saved: 326985.46
ROI: 0.98


In [12]:
high_value = target_df[target_df["customer_value"] > df["customer_value"].median()]

print(f"High Value Customers to Target: {len(high_value)}")

High Value Customers to Target: 0


In [13]:
print(f"Optimal Strategy:")
print(f"- Target Customers: {customers_targeted}")
print(f"- Expected ROI: {roi:.2f}")
print(f"- Focus: High-risk, high-value customers")

Optimal Strategy:
- Target Customers: 331
- Expected ROI: 0.98
- Focus: High-risk, high-value customers


In [14]:
def segment(p):
    if p > 0.7:
        return "High Risk"
    elif p > 0.4:
        return "Medium Risk"
    else:
        return "Low Risk"

df["risk_segment"] = df["churn_probability"].apply(segment)
df["expected_saved"] = df["target"] * df["customer_value"] * RETENTION_SUCCESS_RATE


In [15]:
df_raw = pd.read_csv("../data/raw/Telco-Customer-Churn.csv")

df["customerID"] = df_raw["customerID"]

df[["customerID", "churn_probability"]].head()

,customerID,churn_probability
0,7590-VHVEG,0.842407
1,5575-GNVDE,0.093423
2,3668-QPYBK,0.546508
3,7795-CFOCW,0.102603
4,9237-HQITU,0.879234


In [16]:
df_final = df[[
    "customerID",
    "churn_probability",
    "risk_segment",
    "target",
    "customer_value",
    "expected_saved",
    "Churn",
    "Contract",
    "InternetService",
    "MonthlyCharges"
]]
df_final.head()


,customerID,churn_probability,risk_segment,target,customer_value,expected_saved,Churn,Contract,InternetService,MonthlyCharges
0,7590-VHVEG,0.842407,High Risk,1,29.85,8.955,0,Month-to-month,DSL,29.85
1,5575-GNVDE,0.093423,Low Risk,0,1936.30,0.000,0,One year,DSL,56.95
2,3668-QPYBK,0.546508,Medium Risk,0,107.70,0.000,1,Month-to-month,DSL,53.85
3,7795-CFOCW,0.102603,Low Risk,0,1903.50,0.000,0,One year,DSL,42.30
4,9237-HQITU,0.879234,High Risk,1,141.40,42.420,1,Month-to-month,Fiber optic,70.70


In [17]:
df_final.to_csv("../data/processed/dashboard_data.csv", index=False)

print("Dashboard dataset exported successfully ✅")

Dashboard dataset exported successfully ✅
